#Building a GPT model by following Karpathy's "Let's build a GPT" tutorial

this is a longer,(and admittedly messier) version i wrote to understand language models better. I believe it will be beneficial for people not as proficient in python as me.



In [2]:
#importing text
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text=f.read()

print(len(text))

1115394


In [3]:
#sorting all 'characters' in the file
charss = sorted(list(set(text)))
 #we need to tokenize (convert these into integers)

In [4]:
string_to_integer = {} #defining a dictionary
for i in range(len(charss)):
    ch = charss[i] #assigning each character an integer
    string_to_integer[ch] = i #character is your key and the value is your position
#vice-versa
integer_to_string = {}
for i in range(len(string_to_integer)):
    integer_to_string[i] = charss[i]





In [5]:
#defining the encode and decode functions
def encode(s):
    result =[]
    for i in s: #goes through each character of the string
        result.append(string_to_integer[i])
    return result

def decode(i):
    result = []
    for c in i:
        result.append(integer_to_string[c])
    return ''.join(result) #joins all the characters into a string and returns it

    #testing
print(encode("to be or not to be"))
print(decode(encode("to be or not to be")))




[58, 53, 1, 40, 43, 1, 53, 56, 1, 52, 53, 58, 1, 58, 53, 1, 40, 43]
to be or not to be


As we can see the text has been tokenized(broken down into its characters) into individual numbers, (vectors) and is encoding/decoding correctly.

there are other ways to 'tokenize' the text, (look up tiktoken) but this is the easiest way to get started

we are trading off sequence length for a smaller 'vocabulary' size

In [6]:
#we are now going to tokenize the entire data.

import torch #importing pytorch (very important library for ml)
data = torch.tensor(encode(text), dtype = torch.long) #storing the encoded text as a tensor in pytorch 
#a tensor is an array, but unlike numpy arrays it can remember how its been changed and run on GPUs
#dtype is just defining the datatype.

#print(data.shape,data.dtype)
#print(data[:1000]) #prints the encoding for the first 1000 characters

In [7]:
#going on to seperate our data into a 'training' section and a 'validating' section
n = int(0.9 * len(data)) #first 90% of the data will be used to train our model
train_data = data[:n]
val_data = data[n:]

In [10]:
#we have to use chunks, dont process entire data

block_size = 8 #context length
train_data[:block_size+1]
train_data[:block_size+1].shape



torch.Size([9])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for i in range (block_size):
    context = x[:i+1]
    target = y[i]
    print(f"when input is {context} the target is {target}")

#train using next token

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [ ]:
batch_size = 4
block_size = 8